<!-- track-identity-card -->
# Import of the recorded Assetto Corsa Competizione sessions

| | |
|---|---|
| Pipeline step | `10_user_import.ipynb` |
| Manuscript section | 4.5 |
| Copied from | `notebooks/NB13_user_data_csv_to_parquet_v2.ipynb` |
| Source sha256 | `58ea58cc0618f7bc64ddb4fab7b28b3d` |

**Reads**

- `data/raw/motec_ld/*.csv`

**Writes**

- `data/raw/user_data/standardized/*.parquet`

Converts the MoTeC CSV exports of the first author's sessions into one parquet file per lap, with the channel names mapped onto those of the ACGym recordings. Runs before 11. It is independent of notebooks 01 to 09 and reads none of their outputs.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


In [ ]:
# track-config-bootstrap
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c)); break
else:
    raise RuntimeError("track/config.py not found. Run from inside the repository, or set TRACK_ROOT.")
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


# NB13 — User Data Integration: MoTeC CSV to Pipeline-Ready Parquet (v1)

**Amac:** Efe'nin ACC MoTeC CSV dosyalarini ACGym pipeline'ina uyumlu parquet'lere donustur.

**Veri kaynagi:** `D:\Thesis\data\raw\motec_ld\*.csv` (MoTeC i2 export)

**Kritik ozellik:** CSV'de Distance kolonu var (LD'de yok!) — tur tespiti ve viraj hizalamasi icin kullanilacak.

**Pipeline:** CSV parse → birim standardizasyonu → 50Hz downsample → tur tespiti → parquet

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# =============================================
# KONFIGRASYON
# =============================================

CSV_DIR = str(TRACK_ROOT / "data" / "raw" / "motec_ld")
OUTPUT_DIR = str(TRACK_ROOT / "data" / "raw" / "user_data" / "standardized")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pist uzunluklari (metre)
TRACK_LENGTHS = {
    'monza': 5793,
    'silverstone': 5891,
    'nurburgring': 5148,  # GP layout
}

# Hedef sampling rate
TARGET_HZ = 50

# Minimum tur mesafesi orani (track_length * MIN_LAP_RATIO = minimum gecerli tur)
MIN_LAP_RATIO = 0.92
# Minimum hareket hizi (km/h) — pit/bekleme filtresi
MIN_MOVING_SPEED = 10

# MoTeC CSV header satir sayisi (veri 18. satirdan baslar)
MOTEC_HEADER_ROWS = 17

print("Konfigurasyon yuklendi.")
print(f"CSV dizini: {CSV_DIR}")
print(f"Cikti dizini: {OUTPUT_DIR}")

In [ ]:
# =============================================
# MoTeC CSV Parser
# =============================================

import csv
from io import StringIO

def parse_motec_csv(filepath):
    """MoTeC i2 CSV formatini parse et.
    
    Returns: (metadata_dict, dataframe, units_dict)
    """
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    
    # Metadata (ilk 12 satir) — csv module ile dogru parse
    meta = {}
    for line in lines[:12]:
        row = list(csv.reader(StringIO(line.strip())))
        if row and len(row[0]) >= 2:
            meta[row[0][0]] = row[0][1]
    
    # Kolon isimleri (satir 14, 0-indexed)
    col_row = list(csv.reader(StringIO(lines[14].strip())))
    col_names = col_row[0] if col_row else []
    
    # Birimler (satir 15)
    unit_row = list(csv.reader(StringIO(lines[15].strip())))
    units_list = unit_row[0] if unit_row else []
    units = dict(zip(col_names, units_list))
    
    # Veri (satir 18'den itibaren, 2 bos satir atla)
    df = pd.read_csv(filepath, skiprows=MOTEC_HEADER_ROWS, header=None, 
                      low_memory=False, na_values=[''])
    df.columns = col_names[:len(df.columns)]
    
    # Numerik donusum
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return meta, df, units


# Test — bos dosyalari atla, ilk buyuk dosyayi sec
test_files = [f for f in sorted(os.listdir(CSV_DIR)) 
              if f.endswith('.csv') and f != 'toplu.csv'
              and os.path.getsize(os.path.join(CSV_DIR, f)) > 5000]
print(f"Kullanilabilir CSV sayisi: {len(test_files)}")
meta, df, units = parse_motec_csv(os.path.join(CSV_DIR, test_files[0]))
print(f"\nTest: {test_files[0]}")
print(f"  Venue: {meta.get('Venue', '?')}, Vehicle: {meta.get('Vehicle', '?')}")
print(f"  Shape: {df.shape}, Kolonlar: {len(df.columns)}")
print(f"  SPEED birimi: {units.get('SPEED', '?')}")
print(f"  Distance birimi: {units.get('Distance', '?')}")

In [ ]:
# =============================================
# Standardizasyon: birim donusumu + downsample
# =============================================

def standardize_session(df, units, target_hz=TARGET_HZ):
    """MoTeC CSV verisini pipeline-ready formata donustur.
    
    - Speed: km/h -> m/s
    - Distance: metre (koru)
    - Throttle/Brake: 0-100 (koru)
    - Downsample: 200Hz -> target_hz
    """
    out = pd.DataFrame()
    
    # Time
    out['time'] = df['Time'].values
    
    # Distance (kritik — CSV'den geliyor)
    out['distance'] = df['Distance'].values
    
    # Speed: km/h -> m/s
    speed_unit = units.get('SPEED', 'km/h')
    if 'km' in speed_unit.lower():
        out['speed'] = df['SPEED'].values / 3.6
    else:
        out['speed'] = df['SPEED'].values  # zaten m/s
    
    # Throttle, Brake (0-100)
    out['throttle'] = df['THROTTLE'].values
    out['brake'] = df['BRAKE'].values
    
    # Steering
    out['steer'] = df['STEERANGLE'].values
    
    # Gear
    out['gear'] = df['GEAR'].values
    
    # G-forces
    if 'G_LAT' in df.columns:
        out['g_lat'] = df['G_LAT'].values
    if 'G_LON' in df.columns:
        out['g_lon'] = df['G_LON'].values
    
    # RPM
    if 'RPMS' in df.columns:
        out['rpm'] = df['RPMS'].values
    
    # Wheel speeds (m/s olarak koru)
    for ws in ['WHEEL_SPEED_LF', 'WHEEL_SPEED_RF', 'WHEEL_SPEED_LR', 'WHEEL_SPEED_RR']:
        if ws in df.columns:
            out[ws.lower()] = df[ws].values
    
    # Brake temps
    for bt in ['BRAKE_TEMP_LF', 'BRAKE_TEMP_RF', 'BRAKE_TEMP_LR', 'BRAKE_TEMP_RR']:
        if bt in df.columns:
            out[bt.lower()] = df[bt].values
    
    # Downsample: her N. satiri al
    source_hz = 1.0 / df['Time'].diff().median()
    if source_hz > target_hz * 1.2:
        step = max(1, round(source_hz / target_hz))
        out = out.iloc[::step].reset_index(drop=True)
        actual_hz = 1.0 / out['time'].diff().median()
    else:
        actual_hz = source_hz
    
    return out, actual_hz


# Test
out, hz = standardize_session(df, units)
print(f"Standardize test:")
print(f"  Input: {len(df)} rows")
print(f"  Output: {len(out)} rows @ {hz:.1f} Hz")
print(f"  Kolonlar: {out.columns.tolist()}")
print(f"  Speed: {out['speed'].min():.1f} - {out['speed'].max():.1f} m/s ({out['speed'].max()*3.6:.0f} km/h)")
print(f"  Distance: {out['distance'].min():.0f} - {out['distance'].max():.0f} m")

In [ ]:
# =============================================
# Tur Tespiti: Distance + track_length modulo
# =============================================

def detect_laps(df_std, track_length, min_ratio=MIN_LAP_RATIO):
    """Tam turlari tespit et.
    
    Distance surekli arttigi icin, track_length'e bolup tur numarasi cikarilir.
    Ilk ve son yarim turlar filtrelenir.
    
    Returns: list of (start_idx, end_idx, lap_distance_array) tuples
    """
    dist = df_std['distance'].values
    speed = df_std['speed'].values
    
    # Hareketli bolumu filtrele
    moving = speed > (MIN_MOVING_SPEED / 3.6)  # m/s
    if not moving.any():
        return []
    
    first_move = np.where(moving)[0][0]
    
    # Kumulatif mesafeden tur numarasi
    # Offset: ilk hareket noktasindan itibaren
    dist_from_start = dist - dist[first_move]
    lap_num = (dist_from_start / track_length).astype(int)
    lap_dist = dist_from_start % track_length
    
    # Her turun istatistikleri
    laps = []
    for lap_i in range(lap_num.max() + 1):
        mask = (lap_num == lap_i) & moving
        if mask.sum() < 100:  # cok kisa
            continue
        
        indices = np.where(mask)[0]
        start_idx, end_idx = indices[0], indices[-1]
        
        lap_dist_covered = dist[end_idx] - dist[start_idx]
        lap_time = df_std['time'].iloc[end_idx] - df_std['time'].iloc[start_idx]
        max_speed = speed[mask].max() * 3.6
        min_speed = speed[mask].min() * 3.6
        
        ratio = lap_dist_covered / track_length
        is_complete = min_ratio < ratio < 1.1
        
        laps.append({
            'lap': lap_i,
            'start_idx': start_idx,
            'end_idx': end_idx,
            'distance': lap_dist_covered,
            'time': lap_time,
            'max_speed_kmh': max_speed,
            'min_speed_kmh': min_speed,
            'ratio': ratio,
            'complete': is_complete,
            'rows': mask.sum()
        })
    
    return laps


# Test
track = meta.get('Venue', '').lower().replace(' ', '')
track_len = TRACK_LENGTHS.get(track, 5793)
laps = detect_laps(out, track_len)

print(f"Track: {track} ({track_len}m)")
print(f"Tespit edilen segmentler: {len(laps)}")
print(f"Tam turlar: {sum(1 for l in laps if l['complete'])}")
print()
for lap in laps:
    marker = 'TAM' if lap['complete'] else 'kismi'
    mins = int(lap['time'] // 60)
    secs = lap['time'] % 60
    print(f"  Tur {lap['lap']}: {mins}:{secs:05.2f} | {lap['distance']:.0f}m ({lap['ratio']:.0%}) | "
          f"hiz {lap['min_speed_kmh']:.0f}-{lap['max_speed_kmh']:.0f} km/h | {marker}")

In [ ]:
# =============================================
# TOPLU ISLEM: Tum CSV dosyalarini donustur
# =============================================

csv_files = sorted([f for f in os.listdir(CSV_DIR) 
                     if f.endswith('.csv') and f != 'toplu.csv'
                     and os.path.getsize(os.path.join(CSV_DIR, f)) > 5000])

results = []
all_laps = []

print(f"{'Dosya':55s}  {'Pist':12s}  {'Arac':25s}  {'Rows':>7s}  {'Hz':>5s}  {'Turlar':>6s}")
print("=" * 120)

for csv_name in csv_files:
    filepath = os.path.join(CSV_DIR, csv_name)
    
    try:
        meta, df, units = parse_motec_csv(filepath)
        std, hz = standardize_session(df, units)
        
        track = meta.get('Venue', '').lower().replace(' ', '')
        vehicle = meta.get('Vehicle', '?')
        track_len = TRACK_LENGTHS.get(track, 5793)
        
        laps = detect_laps(std, track_len)
        complete_laps = [l for l in laps if l['complete']]
        
        # Sonuclari kaydet
        for lap in complete_laps:
            lap_data = std.iloc[lap['start_idx']:lap['end_idx']+1].copy()
            # lap_distance: tur icinde 0'dan baslayan mesafe
            lap_data['lap_distance'] = lap_data['distance'] - lap_data['distance'].iloc[0]
            
            lap_id = f"user_{track}_{vehicle.lower().replace(' ', '_')}_{csv_name.replace('.csv', '')}_lap{lap['lap']}"
            lap_info = {
                'file': csv_name, 'track': track, 'vehicle': vehicle,
                'lap_id': lap_id, **lap
            }
            all_laps.append(lap_info)
            
            # Parquet kaydet
            out_path = os.path.join(OUTPUT_DIR, f"{lap_id}.parquet")
            lap_data.to_parquet(out_path, index=False)
        
        n_complete = len(complete_laps)
        total_laps = len(laps)
        print(f"{csv_name:55s}  {track:12s}  {vehicle:25s}  {len(std):6d}  {hz:4.0f}  {n_complete}/{total_laps}")
        
        results.append({
            'file': csv_name, 'track': track, 'vehicle': vehicle,
            'rows': len(std), 'hz': hz, 
            'total_laps': total_laps, 'complete_laps': n_complete
        })
        
    except Exception as e:
        print(f"{csv_name:55s}  HATA: {e}")

print(f"\n{'='*120}")
print(f"Toplam islenmiş: {len(results)} dosya")
print(f"Toplam tam tur: {len(all_laps)}")

In [ ]:
# =============================================
# OZET: Pist x Arac matrisi
# =============================================

if all_laps:
    lap_df = pd.DataFrame(all_laps)
    
    print("=== PIST x ARAC MATRISI (tam tur sayisi) ===\n")
    pivot = lap_df.groupby(['track', 'vehicle']).size().unstack(fill_value=0)
    print(pivot.to_string())
    
    print(f"\n=== TUR ZAMANLARI ===")
    for track in lap_df['track'].unique():
        print(f"\n--- {track.upper()} ---")
        track_laps = lap_df[lap_df['track'] == track]
        for _, row in track_laps.iterrows():
            mins = int(row['time'] // 60)
            secs = row['time'] % 60
            print(f"  {row['vehicle']:25s}  Tur {row['lap']:2d}  {mins}:{secs:05.2f}  "
                  f"hiz {row['min_speed_kmh']:.0f}-{row['max_speed_kmh']:.0f} km/h")
    
    # ACGym karsilastirma icin Monza turlari
    monza_laps = lap_df[lap_df['track'] == 'monza']
    print(f"\n=== MONZA TURLARI (ACGym pipeline icin) ===")
    print(f"Toplam: {len(monza_laps)} tur, {monza_laps['vehicle'].nunique()} farkli arac")
    
    # Parquet ciktilari
    print(f"\n=== URETILEN PARQUET DOSYALARI ===")
    out_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')]
    for f in sorted(out_files):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
        print(f"  {size:5.1f} MB  {f}")
else:
    print("Hic tam tur bulunamadi!")

In [ ]:
print("=" * 60)
print("NB13 User Data Integration v1 — Tamamlandi")
print("=" * 60)
print(f"\nCikti dizini: {OUTPUT_DIR}")
print(f"Toplam parquet: {len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')])}")
print("\nSiradaki adim: NB08 viraj segmentasyonu ile Efe turlarini isle")

In [ ]:
# ============================================================
# ADIM 1: Standardized dizin diagnostik
# ============================================================
from pathlib import Path
import pandas as pd

STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"

files = sorted(STD_DIR.glob("*.parquet"))
print(f"Toplam parquet: {len(files)}\n")

lap_files = []
stint_files = []

for f in files:
    size_kb = f.stat().st_size / 1024
    name = f.name
    if "_lap" in name:
        lap_files.append((name, size_kb))
    else:
        stint_files.append((name, size_kb))

print(f"=== YENi _lap DOSYALARI ({len(lap_files)} adet) ===")
for name, sz in lap_files:
    print(f"  {sz:8.1f} KB  {name}")

print(f"\n=== ESKi STINT DOSYALARI ({len(stint_files)} adet) ===")
for name, sz in stint_files:
    print(f"  {sz:8.1f} KB  {name}")

lap_total = sum(s for _, s in lap_files) / 1024
stint_total = sum(s for _, s in stint_files) / 1024
print(f"\n--- OZET ---")
print(f"_lap  : {len(lap_files):3d} dosya, {lap_total:.1f} MB")
print(f"stint : {len(stint_files):3d} dosya, {stint_total:.1f} MB")
print(f"Toplam: {len(files):3d} dosya, {lap_total + stint_total:.1f} MB")

# Hizli kontrol: _lap dosyalarindan birinin kolonlarini goster
if lap_files:
    sample = pd.read_parquet(STD_DIR / lap_files[0][0])
    print(f"\n--- ORNEK _lap KOLON KONTROLU ({lap_files[0][0]}) ---")
    print(f"Satirlar: {len(sample)}, Kolonlar: {len(sample.columns)}")
    has_dist = "lap_distance" in sample.columns or "Distance" in sample.columns
    print(f"Distance kolonu: {'VAR' if has_dist else 'YOK'}")
    dist_cols = [c for c in sample.columns if "dist" in c.lower() or "distance" in c.lower()]
    print(f"Distance-iliskili kolonlar: {dist_cols}")
    print(f"Ilk 5 kolon: {list(sample.columns[:5])}")

In [ ]:
# ============================================================
# ADIM 1b: Kolon uyumluluk kontrolu + stint silme onay
# ============================================================
from pathlib import Path
import pandas as pd

STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"
T1_DIR  = TRACK_ROOT / "data" / "processed" / "tier1_small_3t1c"

# --- Efe _lap kolonlari ---
lap_sample = sorted(STD_DIR.glob("*_lap*.parquet"))[0]
efe_cols = set(pd.read_parquet(lap_sample).columns)
print(f"Efe _lap kolonlari ({len(efe_cols)}):")
print(f"  {sorted(efe_cols)}\n")

# --- ACGym T1 kolonlari ---
t1_files = sorted(T1_DIR.glob("*.parquet"))
if t1_files:
    acgym_cols = set(pd.read_parquet(t1_files[0]).columns)
    print(f"ACGym T1 kolonlari ({len(acgym_cols)}):")
    print(f"  {sorted(acgym_cols)[:20]}...")  # ilk 20
    
    # Kesisim ve farklar
    common = efe_cols & acgym_cols
    only_efe = efe_cols - acgym_cols
    only_acgym = acgym_cols - efe_cols
    print(f"\nOrtak kolonlar ({len(common)}): {sorted(common)}")
    print(f"Sadece Efe ({len(only_efe)}): {sorted(only_efe)}")
    print(f"Sadece ACGym ({len(only_acgym)}): {sorted(only_acgym)[:15]}...")

# --- NB08-NB09 icin kritik kolonlar ---
CRITICAL = ["speed", "throttle", "brake", "steer", "lap_distance"]
missing = [c for c in CRITICAL if c not in efe_cols]
print(f"\nNB08-NB09 kritik kolonlar: {CRITICAL}")
print(f"Eksik: {missing if missing else 'YOK - TAMAM'}")

# --- Stint dosyalari listesi (silme icin) ---
stint_files = sorted([f for f in STD_DIR.glob("*.parquet") if "_lap" not in f.name])
print(f"\n--- SiLiNECEK STINT DOSYALARI ({len(stint_files)} adet, {sum(f.stat().st_size for f in stint_files)/1024/1024:.1f} MB) ---")

In [ ]:
# ============================================================
# ADIM 2: Stint temizlik + kolon esleme haritasi
# ============================================================
from pathlib import Path
import pandas as pd

STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"
T1_DIR  = TRACK_ROOT / "data" / "processed" / "tier1_small_3t1c"

# --- 2a: ACGym kolon isimlerini tam listele ---
t1_sample = pd.read_parquet(sorted(T1_DIR.glob("*.parquet"))[0])
print("ACGym T1 tum kolonlar:")
for i, c in enumerate(sorted(t1_sample.columns)):
    print(f"  {i+1:2d}. {c}")

# --- 2b: NB08-NB09 icin ACGym'de hangi kolonlar kullaniliyor? ---
# NB08 state machine kritik: speed, brake, throttle, steer, lap_distance
# NB09 fingerprint: ayni + rpm, gear, g_lat, g_lon, wheel_speeds
EFE_TO_ACGYM = {
    "speed":       "speed",         # ayni isim
    "throttle":    "Gas",
    "brake":       "brakeStatus",
    "steer":       "steerAngle",
    "lap_distance":"LapDist",
    "rpm":         "RPM",
    "gear":        "actualGear",
    "g_lat":       "accelX",        # lateral g
    "g_lon":       "accelY",        # longitudinal g
}
print(f"\nOnerilen esleme ({len(EFE_TO_ACGYM)} kolon):")
for efe_name, acgym_name in EFE_TO_ACGYM.items():
    in_efe = efe_name in pd.read_parquet(sorted(STD_DIR.glob("*_lap*.parquet"))[0]).columns
    in_acgym = acgym_name in t1_sample.columns
    status = "OK" if (in_efe and in_acgym) else "KONTROL"
    print(f"  {efe_name:15s} -> {acgym_name:15s}  [{status}]")

# --- 2c: Stint silme ---
stint_files = sorted([f for f in STD_DIR.glob("*.parquet") if "_lap" not in f.name])
print(f"\n--- STINT SILME ---")
for f in stint_files:
    f.unlink()
    print(f"  SILINDI: {f.name}")
print(f"\nKalan dosyalar: {len(list(STD_DIR.glob('*.parquet')))}")

In [ ]:
# ============================================================
# ADIM 3: Throttle kolon dogrulama + birim kontrolu
# ============================================================
from pathlib import Path
import pandas as pd

T1_DIR = TRACK_ROOT / "data" / "processed" / "tier1_small_3t1c"
STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"

t1 = pd.read_parquet(sorted(T1_DIR.glob("*.parquet"))[0])
efe = pd.read_parquet(sorted(STD_DIR.glob("*_lap*.parquet"))[0])

# --- accStatus throttle mi? ---
print("=== ACGym accStatus istatistikleri ===")
print(t1["accStatus"].describe())
print(f"Min: {t1['accStatus'].min():.4f}, Max: {t1['accStatus'].max():.4f}")

print("\n=== Efe throttle istatistikleri ===")
print(efe["throttle"].describe())
print(f"Min: {efe['throttle'].min():.4f}, Max: {efe['throttle'].max():.4f}")

# --- brakeStatus vs brake ayni aralik mi? ---
print("\n=== ACGym brakeStatus ===")
print(f"Min: {t1['brakeStatus'].min():.4f}, Max: {t1['brakeStatus'].max():.4f}")
print(f"\n=== Efe brake ===")
print(f"Min: {efe['brake'].min():.4f}, Max: {efe['brake'].max():.4f}")

# --- speed birimi kontrolu ---
print(f"\n=== Speed birimi ===")
print(f"ACGym speed  : mean={t1['speed'].mean():.1f}, max={t1['speed'].max():.1f}")
print(f"ACGym speed_kmh: mean={t1['speed_kmh'].mean():.1f}, max={t1['speed_kmh'].max():.1f}")
print(f"Efe speed    : mean={efe['speed'].mean():.1f}, max={efe['speed'].max():.1f}")

# --- steer aralik ---
print(f"\n=== Steer aralik ===")
print(f"ACGym steerAngle: min={t1['steerAngle'].min():.3f}, max={t1['steerAngle'].max():.3f}")
print(f"Efe steer      : min={efe['steer'].min():.3f}, max={efe['steer'].max():.3f}")

# --- LapDist vs lap_distance ---
print(f"\n=== Mesafe aralik ===")
print(f"ACGym LapDist    : min={t1['LapDist'].min():.1f}, max={t1['LapDist'].max():.1f}")
print(f"Efe lap_distance : min={efe['lap_distance'].min():.1f}, max={efe['lap_distance'].max():.1f}")

In [ ]:
# ============================================================
# ADIM 4: Distance offset hizalama kontrolu (Monza)
# ============================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

T1_DIR  = TRACK_ROOT / "data" / "processed" / "tier1_small_3t1c"
STD_DIR = TRACK_ROOT / "data" / "raw" / "user_data" / "standardized"

# --- ACGym Monza ornegi ---
t1_files = sorted(T1_DIR.glob("*monza*bmw*.parquet"))
acgym = pd.read_parquet(t1_files[0])
# Tek bir tur sec (LapDist 0'a yakin reset noktasi bul)
resets = acgym.index[acgym["LapDist"].diff() < -1000]
if len(resets) > 1:
    acgym_lap = acgym.iloc[resets[0]:resets[1]].copy()
else:
    acgym_lap = acgym.iloc[:resets[0]].copy() if len(resets) > 0 else acgym.copy()

# --- Efe Monza ornegi (GTR, en cok tur) ---
efe_files = sorted(STD_DIR.glob("*monza_gtr*_lap0.parquet"))
efe_lap = pd.read_parquet(efe_files[0])

# --- Frenleme profili karsilastirmasi ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# ACGym
ax1 = axes[0]
ax1.plot(acgym_lap["LapDist"], acgym_lap["brakeStatus"], 
         color="tab:red", alpha=0.7, linewidth=0.5)
ax1.set_ylabel("Brake (0-1)")
ax1.set_title(f"ACGym Monza BMW Z4 - Frenleme Profili (1 tur, {len(acgym_lap)} satir)")
ax1.set_xlim(0, 5800)
ax1.grid(True, alpha=0.3)

# Efe
ax2 = axes[1]
ax2.plot(efe_lap["lap_distance"], efe_lap["brake"] / 100, 
         color="tab:blue", alpha=0.7, linewidth=0.5)
ax2.set_ylabel("Brake (0-1)")
ax2.set_xlabel("Distance (m)")
ax2.set_title(f"Efe Monza Nissan GT-R - Frenleme Profili (lap0, {len(efe_lap)} satir)")
ax2.set_xlim(0, 5800)
ax2.grid(True, alpha=0.3)

# NB07 viraj referans cizgileri (Monza ilk 5 viraj)
# Bunlari NB07 ciktisindan alacagiz, simdilik gorsel karsilastirma
plt.tight_layout()
plt.savefig(str(TRACK_ROOT / "results" / "figures" / "brake_alignment_check.png"), dpi=150)
plt.show()

print(f"\nACGym LapDist: {acgym_lap['LapDist'].min():.1f} - {acgym_lap['LapDist'].max():.1f} m")
print(f"Efe lap_dist : {efe_lap['lap_distance'].min():.1f} - {efe_lap['lap_distance'].max():.1f} m")
print(f"\nACGym ilk frenleme (brake>0.1): {acgym_lap.loc[acgym_lap['brakeStatus']>0.1, 'LapDist'].iloc[0]:.0f} m")
print(f"Efe   ilk frenleme (brake>10) : {efe_lap.loc[efe_lap['brake']>10, 'lap_distance'].iloc[0]:.0f} m")

In [ ]:
# ============================================================
# ADIM 5: NB08 + NB09 fonksiyon envanteri
# Hangi fonksiyonlar/siniflar tanimli, NB14'e ne tasimamiz lazim
# ============================================================
from pathlib import Path
import json

NB_DIR = TRACK_ROOT / "notebooks"

for nb_name in ["NB08_corner_segmentation_v4.ipynb", 
                "NB09_fingerprint_v3.ipynb",
                "NB10A_clustering_validation_v1.ipynb"]:
    nb_path = NB_DIR / nb_name
    if not nb_path.exists():
        # v3 fallback
        alt = nb_name.replace("_v4", "_v3")
        nb_path = NB_DIR / alt
    if not nb_path.exists():
        print(f"BULUNAMADI: {nb_name}")
        continue
    
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    
    code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]
    print(f"\n{'='*60}")
    print(f"{nb_path.name} — {len(code_cells)} code cell")
    print(f"{'='*60}")
    
    for i, cell in enumerate(code_cells):
        src = "".join(cell["source"])
        # def ve class tanimlarini bul
        defs = [line.strip() for line in src.split("\n") 
                if line.strip().startswith("def ") or line.strip().startswith("class ")]
        # Onemli degiskenleri bul (BUYUK HARF sabitleri)
        consts = [line.split("=")[0].strip() for line in src.split("\n")
                  if "=" in line and line.split("=")[0].strip().isupper() 
                  and not line.strip().startswith("#")]
        
        if defs or consts:
            print(f"\n  Cell {i}:")
            for d in defs:
                print(f"    {d}")
            for c in consts[:5]:  # ilk 5 sabit
                print(f"    CONST: {c}")

In [ ]:
# ============================================================
# ADIM 6: Kritik fonksiyon govdeleri (NB14 icin)
# ============================================================
from pathlib import Path
import json

NB_DIR = TRACK_ROOT / "notebooks"

extract = {
    "NB08_corner_segmentation_v4.ipynb": [0, 2, 3, 4, 5, 7],
    "NB09_fingerprint_v3.ipynb": [0, 1, 2, 3, 5, 6],
}

for nb_name, cells_wanted in extract.items():
    nb_path = NB_DIR / nb_name
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    
    code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]
    print(f"\n{'#'*70}")
    print(f"# {nb_name}")
    print(f"{'#'*70}")
    
    for idx in cells_wanted:
        if idx < len(code_cells):
            src = "".join(code_cells[idx]["source"])
            print(f"\n# --- Cell {idx} ---")
            print(src)
            print(f"# --- End Cell {idx} ---")

In [ ]:
# ============================================================
# ADIM 7: corners_v3 + T1 fingerprint/scaler konumu
# ============================================================
from pathlib import Path
import pandas as pd

PROJECT = TRACK_ROOT

# corners_v3 arama
search_dirs = [
    PROJECT / "data" / "features",
    PROJECT / "data" / "processed" / "acgym_sessions",
    PROJECT / "results",
    PROJECT / "notebooks" / "results",
]

print("=== CORNERS_V3 ARAMA ===")
for d in search_dirs:
    if not d.exists():
        continue
    for f in sorted(d.rglob("*corner*")):
        sz = f.stat().st_size / 1024
        print(f"  {sz:8.1f} KB  {f.relative_to(PROJECT)}")

# fingerprints + scaler arama
print("\n=== FINGERPRINT / SCALER ARAMA ===")
fp_dirs = [
    PROJECT / "data" / "fingerprints",
    PROJECT / "data" / "features",
]
for d in fp_dirs:
    if not d.exists():
        continue
    for f in sorted(d.glob("*")):
        if f.is_file():
            sz = f.stat().st_size / 1024
            print(f"  {sz:8.1f} KB  {f.relative_to(PROJECT)}")

# T1 features ornegi
print("\n=== T1 FEATURES KOLON KONTROL ===")
feat_dir = PROJECT / "data" / "features"
for f in sorted(feat_dir.glob("driver_corner_matrix_monza*")):
    df = pd.read_parquet(f)
    print(f"{f.name}: {len(df)} suruku x {len(df.columns)} kolon")
    print(f"  Kolonlar: {list(df.columns[:10])}...")
    print(f"  driver_ids: {sorted(df['driver_id'].unique())[:5]}...")